In [1]:
import sys
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    """
    Find the project root directory by looking for a `pyproject.toml` file.

    Args:
        start (Path | None): Optional start path

    Returns:
        Path: The project root directory

    Raises:
        RuntimeError: If the project root cannot be found
    """
    current = (start or Path.cwd()).resolve()

    for parent in [current, *current.parents]:
        if (parent / "pyproject.toml").exists():
            return parent

    raise RuntimeError("Could not locate project root")


PROJECT_ROOT = find_project_root()

SOURCE_DIR = PROJECT_ROOT / "src"
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

print(f"Project root: {PROJECT_ROOT}")
print(f"Source directory: {SOURCE_DIR}")

Project root: /Users/brenoingwersensantos/Documents/GitHub/vrptw-lab
Source directory: /Users/brenoingwersensantos/Documents/GitHub/vrptw-lab/src


In [2]:
from pathlib import Path

import pandas as pd

In [3]:
def _normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalize the column names to snake case and remove periods.
    """
    df.columns = [
        col.strip().lower().replace(" ", "_").replace(".", "") for col in df.columns
    ]
    return df


def load_solomon_data(path: Path):
    """
    Load Solomon data from a CSV file.
    """
    df = _normalize_columns(pd.read_csv(path))
    return df

In [4]:
name = "solomon"
instance_file = "C1/C101.csv"

In [5]:
path = Path(f"../data/{name}/{instance_file}")
df = load_solomon_data(path)
print(df.shape)
df.head()

(101, 7)


,cust_no,xcoord,ycoord,demand,ready_time,due_date,service_time
0,1,40,50,0,0,1236,0
1,2,45,68,10,912,967,90
2,3,45,70,30,825,870,90
3,4,42,66,10,65,146,90
4,5,42,68,10,727,782,90


In [6]:
from vrptw.instance import VRPTWInstance
from vrptw.problem import VRPTWProblem
from vrptw.solver import VRPTWSolver
from vrptw.solver_config import SolverConfig

instance = VRPTWInstance.from_df(df)
problem = VRPTWProblem(instance=instance, max_trucks=20, truck_capacity=200)
config = SolverConfig(max_time_in_seconds=90)
solver = VRPTWSolver(problem, config)

2026-09-13 23:43:49.691 | INFO     | vrptw.variables:__init__:23 - Creating the model variables...
2026-09-13 23:43:49.691 | INFO     | vrptw.variables:__init__:24 - Creating the arc variables...
2026-09-13 23:43:49.724 | INFO     | vrptw.variables:__init__:29 - Creating the node load variables...
2026-09-13 23:43:49.725 | INFO     | vrptw.variables:__init__:36 - Creating the node start time variables...
2026-09-13 23:43:49.725 | INFO     | vrptw.solver:_add_constraints:92 - Adding constraints to the model...
2026-09-13 23:43:49.726 | INFO     | vrptw.solver:_add_circuit_constraint:108 - Constraint: ensure that each node is visited (multiple circuits allowed).
2026-09-13 23:43:49.733 | INFO     | vrptw.solver:_add_max_trucks_constraint:118 - Constraint: limit the number of trucks to 20.
2026-09-13 23:43:49.734 | INFO     | vrptw.solver:_add_load_constraint:127 - Constraint: limit the load on each truck route.
2026-09-13 23:43:49.774 | INFO     | vrptw.solver:_add_time_window_constraint

In [7]:
result = solver.solve()

2026-09-13 23:43:49.850 | INFO     | vrptw.solver:solve:285 - Solving the VRPTW problem...
2026-09-13 23:43:49.850 | INFO     | vrptw.solver:_solve_stage1_minimize_trucks:252 - Stage 1: minimizing trucks...
2026-09-13 23:43:49.851 | INFO     | vrptw.solver:_add_truck_minimization_objective:166 - Objective: minimize the number of trucks used
2026-09-13 23:43:50.316 | INFO     | vrptw.callback:on_solution_callback:15 - Solution found with objective: 10.0
2026-09-13 23:43:50.373 | SUCCESS  | vrptw.solver:_build_solve_result:228 - Solver finished with status: OPTIMAL


In [8]:
print(f"Status: {result.status_name}")
print(f"Trucks: {result.n_trucks}")
print(f"Runtime: {result.runtime_seconds:.2f}s")
print(f"Solver status: {solver.status_name}")

Status: OPTIMAL
Trucks: 10
Runtime: 0.52s
Solver status: OPTIMAL


In [9]:
stops_df = result.solution.to_stops_df(instance)
stops_df.head()

,truck_id,sequence,cust_no_from,cust_no_to
0,1,1,1,6
1,1,2,6,25
2,1,3,25,26
3,1,4,26,28
4,1,5,28,30


In [10]:
break

SyntaxError: 'break' outside loop (668683560.py, line 1)

In [ ]:
stops_df = pd.merge(
    result.solution.to_stops_df(instance),
    df[["cust_no", "demand", "ready_time", "due_date", "service_time"]],
    left_on="cust_no_to",
    right_on="cust_no",
    how="left",
    validate="many_to_one"
).drop(columns=["cust_no"])
stops_df.head()

In [ ]:
g = stops_df.groupby("truck_id").agg(
    total_stops=("sequence", "max"),
    start_from=("cust_no_from", "first"),
    end_at=("cust_no_to", "last"),
    total_demand=("demand", "sum"),
    expected_travel_time=("distance", "sum"),
    expected_service_time=("service_time", "sum"),
)
g["total_time"] = g["expected_travel_time"] + g["expected_service_time"]
g.head()